In [2]:
import os 
from pathlib import Path
import pandas as pd 


In [3]:
# os.chdir(r"C:\Full_stack\Portfolios\Projets\Data_Analysis_Projects\rte")
PROJECT_DIR = Path(r"C:\Full_stack\Portfolios\Projets\Data_Analysis_Projects\rte")
RAW_DATA = PROJECT_DIR / "data" / "raw" / "Bretagne"


In [4]:
all_data = []

for data in RAW_DATA.iterdir():
    df = pd.read_csv(
        data,
        sep="\t",
        encoding="latin1",
        dtype=str,
        na_values=["-", "", "NA", "N/A"],
        keep_default_na=True,
        index_col= False
    )

    df = df.iloc[:-1].copy()

    df = df.loc[:, ~df.columns.str.startswith("Unnamed")].copy()

    df["datetime"] = pd.to_datetime(
        df["Date"] + " " + df["Heures"], errors="coerce", dayfirst=True
    )

    df = df[df["datetime"].dt.minute.isin([0, 30])].copy()

    all_data.append(df)


C:\Users\user\AppData\Local\Temp\ipykernel_37032\122982535.py:4: ParserWarning: Length of header or names does not match length of data. This leads to a loss of data with index_col=False.
  df = pd.read_csv(


In [5]:
reference_columns = set(all_data[0].columns)

for i, data in enumerate(all_data):
    current_columns = set(data.columns)

    missing = reference_columns - current_columns
    extra = current_columns - reference_columns

    print(f"\nDataFrame {all_data[i]['Date'].str[:4].iloc[0]}")

    if not missing and not extra:
        print("✅ Même variables")
    else:
        print("❌ Variables différentes")
        print("Colonnes manquantes :", missing)
        print("Colonnes supplémentaires :", extra)



DataFrame 2013
✅ Même variables

DataFrame 2014
✅ Même variables

DataFrame 2015
✅ Même variables

DataFrame 2016
✅ Même variables

DataFrame 2017
✅ Même variables

DataFrame 2018
✅ Même variables

DataFrame 2019
✅ Même variables

DataFrame 2020
❌ Variables différentes
Colonnes manquantes : set()
Colonnes supplémentaires : {'Flux physiques de Nouvelle-Aquitaine vers Bretagne', 'Flux physiques de Bretagne vers Allemagne', 'TCO Thermique (%)', 'TCO Nucléaire (%)', 'TCH Nucléaire (%)', 'TCH Thermique (%)', 'Flux physiques de Centre-Val de Loire vers Bretagne', 'Flux physiques de Bretagne vers Ile-de-France', 'Flux physiques Belgique vers Bretagne', 'Flux physiques de Bretagne vers Centre-Val de Loire', 'Flux physiques de Bretagne vers Italie', 'TCH Hydraulique (%)', 'Flux physiques de Bretagne vers Bretagne.1', 'Flux physiques de Bretagne vers Royaume-Uni', 'Flux physiques de Bretagne vers Occitanie', 'Flux physiques de Bretagne vers Luxembourg', 'Flux physiques de Bourgogne-Franche-Comt

In [6]:
os.chdir(PROJECT_DIR)
# os.getcwd()

In [7]:
# # all_data[6] # 6
# df = pd.read_csv(r"data\raw\Bretagne\eCO2mix_RTE_Bretagne_Annuel-Definitif_2019.xls" ,
#             sep ="\t" ,
#             encoding= "latin1",
#             index_col =False)

# print(df.index[:5])
# print(df.columns.tolist())


In [8]:
df = pd.concat(all_data , ignore_index =False , sort = False)

In [9]:
pd.set_option("Display.max_rows",99)
pd.set_option("Display.max_columns", 99)


In [10]:
(df.isna().sum()).sort_values(ascending = False) / len(df)

Flux physiques de Grand-Est vers Bretagne                  1.000000
Flux physiques de Hauts-de-France vers Bretagne            1.000000
Flux physiques de Centre-Val de Loire vers Bretagne        1.000000
Flux physiques de Bretagne vers Bretagne                   1.000000
Flux physiques Suisse vers Bretagne                        1.000000
Flux physiques de Bretagne vers Allemagne                  1.000000
Flux physiques de Bretagne vers Suisse                     1.000000
Flux physiques de Bretagne vers Royaume-Uni                1.000000
Flux physiques d'Auvergne-Rhône-Alpes vers Bretagne        1.000000
Flux physiques de Bourgogne-Franche-Comté vers Bretagne    1.000000
Flux physiques de Bretagne vers Nouvelle-Aquitaine         1.000000
Flux physiques de Bretagne vers Ile-de-France              1.000000
Flux physiques de Bretagne vers Bretagne.1                 1.000000
Flux physiques de Bretagne vers Centre-Val de Loire        1.000000
Flux physiques de Bretagne vers Grand-Est       

In [11]:
df = df.dropna(axis=1 , how="all")#.isna().sum().sort_values(ascending=False) / len(df)

In [12]:
ENERGY_COLUMNS = [
    "Consommation",
    "Thermique",
    "Nucléaire",
    "Eolien",
    "Solaire",
    "Hydraulique",
    "Pompage",
    "Bioénergies",
    "Ech. physiques",
    "Stockage batterie",
    "Déstockage batterie",
    "Eolien terrestre",
    "Eolien offshore",
]


DIM_COLUMNS = [
    "Périmètre",
    "Nature",
    "Date",
    "Heures",
    "datetime",
]

In [13]:
FLOW_COLUMNS = df.columns[df.columns.str.contains("Flux", regex=True)].tolist()

In [14]:
RATE_COLUMNS = [col for col in df.columns if col.startswith(("TCO", "TCH"))]


In [15]:
print("Dimensions :", DIM_COLUMNS)
print("Energy :", ENERGY_COLUMNS)
print("Flows :", FLOW_COLUMNS)
print("Rates :", RATE_COLUMNS)


Dimensions : ['Périmètre', 'Nature', 'Date', 'Heures', 'datetime']
Energy : ['Consommation', 'Thermique', 'Nucléaire', 'Eolien', 'Solaire', 'Hydraulique', 'Pompage', 'Bioénergies', 'Ech. physiques', 'Stockage batterie', 'Déstockage batterie', 'Eolien terrestre', 'Eolien offshore']
Flows : ['Flux physiques de Normandie vers Bretagne', "Flux physiques d'Occitanie vers Bretagne", 'Flux physiques de Pays-de-la-Loire vers Bretagne', 'Flux physiques de Bretagne vers Auvergne-Rhône-Alpes', 'Flux physiques de Bretagne vers Normandie', 'Flux physiques de Bretagne vers Occitanie', 'Flux physiques de Bretagne vers Pays-de-la-Loire', 'Flux physiques Allemagne vers Bretagne']
Rates : ['TCO Thermique (%)', 'TCH Thermique (%)', 'TCO Nucléaire (%)', 'TCH Nucléaire (%)', 'TCO Eolien (%)', 'TCH Eolien (%)', 'TCO Solaire (%)', 'TCH Solaire (%)', 'TCO Hydraulique (%)', 'TCH Hydraulique (%)', 'TCO Bioénergies (%)', 'TCH Bioénergies (%)']


In [16]:
analytics_columns = DIM_COLUMNS + ENERGY_COLUMNS
analytics_df = df[[col for col in analytics_columns if col in df.columns]].copy()
flows_df = df[["datetime", "Périmètre"] + FLOW_COLUMNS].copy()
rates_df = df[["datetime", "Périmètre"] + RATE_COLUMNS].copy()

filename ={
    "bretagne_analysis": analytics_df ,
    "bretagne_flows":flows_df ,
    "bretagne_rates_energy" :rates_df }

for file , data in filename.items() :
    data.to_parquet(f"data/staging/{file}.parquet" , index=False )# , engine ="pyarrow")